# 🚗 Vehicle Damage Detection & Inspection System

### PyTorch + YOLO + Computer Vision

A computer vision system for detecting and localizing visible vehicle
damage from vehicle images using object detection.

## Objective

Detect vehicle damage using bounding boxes and confidence scores
and generate structured inspection results.

## Technology

- Python
- PyTorch
- YOLO
- OpenCV
- NumPy
- Pandas
- Google Colab
- Kaggle

## Pipeline

Dataset
→ Data Validation
→ Preprocessing
→ YOLO Training
→ Evaluation
→ Inference
→ Artifact Preservation
→ Productionization

In [1]:
!pip install -q ultralytics kaggle opencv-python-headless pyyaml

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.7/45.7 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 75.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.5/63.5 kB 7.2 MB/s eta 0:00:00


In [2]:
import os
import json
import random
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import torch

from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.


In [3]:
print("Python version:", platform.python_version())
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)

Python version: 3.13.15
PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
CUDA version: 12.8


In [4]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Random seed set to: {SEED}")

Random seed set to: 42


In [5]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Training device:", DEVICE)

Training device: cuda


In [6]:
PROJECT_ROOT = Path("/content/vehicle_damage_detection")

DATA_DIR = PROJECT_ROOT / "data"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

MODELS_DIR = ARTIFACTS_DIR / "models"
METRICS_DIR = ARTIFACTS_DIR / "metrics"
PLOTS_DIR = ARTIFACTS_DIR / "plots"
PREDICTIONS_DIR = ARTIFACTS_DIR / "predictions"
CONFIG_DIR = ARTIFACTS_DIR / "configs"
METADATA_DIR = ARTIFACTS_DIR / "metadata"

for directory in [
    PROJECT_ROOT,
    DATA_DIR,
    ARTIFACTS_DIR,
    MODELS_DIR,
    METRICS_DIR,
    PLOTS_DIR,
    PREDICTIONS_DIR,
    CONFIG_DIR,
    METADATA_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

print("Project directories created successfully.")

Project directories created successfully.


Before proceeding, please ensure you have your Kaggle API token stored in Colab secrets with the name `KAGGLE_TOKEN`. You can add it by clicking the '🔑' icon in the left panel.

In [7]:
from google.colab import userdata
import os
import json
from pathlib import Path

# Get Kaggle API token and username from Colab secrets
kaggle_username = userdata.get('KAGGLE_USERNAME')
kaggle_key = userdata.get('KAGGLE_KEY')

# Create the .kaggle directory if it doesn't exist
os.makedirs(Path.home() / '.kaggle', exist_ok=True)

# Write the kaggle.json file
with open(Path.home() / '.kaggle' / 'kaggle.json', 'w') as f:
    json.dump({'username': kaggle_username, 'key': kaggle_key}, f)

# Set permissions for kaggle.json
os.chmod(Path.home() / '.kaggle' / 'kaggle.json', 0o600)

print("Kaggle credentials set up successfully.")

Kaggle credentials set up successfully.


In [8]:
!kaggle datasets download \
    -d gabrielfcarvalho/cardd-with-yolo-annotations-images-labels \
    -p /content/vehicle_damage_detection/data

Dataset URL: https://www.kaggle.com/datasets/gabrielfcarvalho/cardd-with-yolo-annotations-images-labels
License(s): Attribution-NonCommercial 4.0 International (CC BY-NC 4.0)
100% 2.80G/2.80G [02:16<00:00, 22.1MB/s]



In [9]:
!find /content/vehicle_damage_detection/data -maxdepth 2 -type f

/content/vehicle_damage_detection/data/cardd-with-yolo-annotations-images-labels.zip


In [10]:
import zipfile

zip_path = DATA_DIR / "cardd-with-yolo-annotations-images-labels.zip"
EXTRACT_DIR = DATA_DIR / "carddd"

EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

print(f"Dataset extracted to: {EXTRACT_DIR}")

Dataset extracted to: /content/vehicle_damage_detection/data/carddd


In [11]:
for path in EXTRACT_DIR.iterdir():
    print(path)

/content/vehicle_damage_detection/data/carddd/data.yaml
/content/vehicle_damage_detection/data/carddd/test
/content/vehicle_damage_detection/data/carddd/train
/content/vehicle_damage_detection/data/carddd/val


In [12]:
for path in sorted(EXTRACT_DIR.rglob("*")):
    relative_path = path.relative_to(EXTRACT_DIR)

    if path.is_dir():
        print(f"[DIR]  {relative_path}")
    else:
        print(f"[FILE] {relative_path}")

Streaming output truncated to the last 5000 lines.
[FILE] train/images/003194.jpg
[FILE] train/images/003195.jpg
[FILE] train/images/003197.jpg
[FILE] train/images/003198.jpg
[FILE] train/images/003199.jpg
[FILE] train/images/003202.jpg
[FILE] train/images/003207.jpg
[FILE] train/images/003208.jpg
[FILE] train/images/003209.jpg
[FILE] train/images/003210.jpg
[FILE] train/images/003211.jpg
[FILE] train/images/003212.jpg
[FILE] train/images/003215.jpg
[FILE] train/images/003216.jpg
[FILE] train/images/003217.jpg
[FILE] train/images/003218.jpg
[FILE] train/images/003219.jpg
[FILE] train/images/003220.jpg
[FILE] train/images/003221.jpg
[FILE] train/images/003223.jpg
[FILE] train/images/003224.jpg
[FILE] train/images/003225.jpg
[FILE] train/images/003227.jpg
[FILE] train/images/003228.jpg
[FILE] train/images/003229.jpg
[FILE] train/images/003230.jpg
[FILE] train/images/003231.jpg
[FILE] train/images/003232.jpg
[FILE] train/images/003236.jpg
[FILE] train/images/003237.jpg
[FILE] train/images

In [13]:
import yaml

yaml_path = EXTRACT_DIR / "data.yaml"

with open(yaml_path, "r") as file:
    data_config = yaml.safe_load(file)

data_config

{'train': '/kaggle/input/cardd-with-yolo-annotations-images-labels/train/images',
 'val': '/kaggle/input/cardd-with-yolo-annotations-images-labels/val/images',
 'test': '/kaggle/input/cardd-with-yolo-annotations-images-labels/test/images',
 'nc': 6,
 'names': ['dent',
  'scratch',
  'crack',
  'glass shatter',
  'lamp broken',
  'tire flat']}

In [14]:
print("Classes:", data_config.get("names"))
print("Number of classes:", data_config.get("nc"))
print("Train path:", data_config.get("train"))
print("Validation path:", data_config.get("val"))
print("Test path:", data_config.get("test"))

Classes: ['dent', 'scratch', 'crack', 'glass shatter', 'lamp broken', 'tire flat']
Number of classes: 6
Train path: /kaggle/input/cardd-with-yolo-annotations-images-labels/train/images
Validation path: /kaggle/input/cardd-with-yolo-annotations-images-labels/val/images
Test path: /kaggle/input/cardd-with-yolo-annotations-images-labels/test/images


In [15]:
for split in ["train", "val", "test"]:
    split_dir = EXTRACT_DIR / split

    image_dir = split_dir / "images"
    label_dir = split_dir / "labels"

    print(f"\n{split.upper()}")
    print("Images directory exists:", image_dir.exists())
    print("Labels directory exists:", label_dir.exists())


TRAIN
Images directory exists: True
Labels directory exists: True

VAL
Images directory exists: True
Labels directory exists: True

TEST
Images directory exists: True
Labels directory exists: True


In [16]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

for split in ["train", "val", "test"]:
    image_dir = EXTRACT_DIR / split / "images"
    label_dir = EXTRACT_DIR / split / "labels"

    images = [
        p for p in image_dir.iterdir()
        if p.suffix.lower() in IMAGE_EXTENSIONS
    ]

    labels = list(label_dir.glob("*.txt"))

    print(f"\n{split.upper()}")
    print("Images:", len(images))
    print("Labels:", len(labels))


TRAIN
Images: 2816
Labels: 2816

VAL
Images: 810
Labels: 810

TEST
Images: 374
Labels: 374


In [17]:
sample_label = next((EXTRACT_DIR / "train" / "labels").glob("*.txt"))

print("Sample label file:", sample_label)
print("\nContents:\n")

with open(sample_label, "r") as file:
    print(file.read())

Sample label file: /content/vehicle_damage_detection/data/carddd/train/labels/000280.txt

Contents:

3 0.548313 0.434210 0.602684 0.193540



In [18]:
from collections import Counter

class_counts = Counter()

for label_file in (EXTRACT_DIR / "train" / "labels").glob("*.txt"):
    with open(label_file, "r") as file:
        for line in file:
            line = line.strip()

            if not line:
                continue

            class_id = int(line.split()[0])
            class_counts[class_id] += 1

print("Training object distribution:")
print(class_counts)

Training object distribution:
Counter({1: 2560, 0: 1806, 2: 651, 4: 494, 3: 475, 5: 225})


In [19]:
class_names = data_config["names"]

print("Class mapping:")

for class_id, class_name in enumerate(class_names):
    print(f"{class_id}: {class_name}")

Class mapping:
0: dent
1: scratch
2: crack
3: glass shatter
4: lamp broken
5: tire flat


In [20]:
invalid_boxes = []

for split in ["train", "val", "test"]:
    label_dir = EXTRACT_DIR / split / "labels"

    for label_file in label_dir.glob("*.txt"):
        with open(label_file, "r") as file:
            for line_number, line in enumerate(file, start=1):
                line = line.strip()

                if not line:
                    continue

                values = line.split()

                if len(values) != 5:
                    invalid_boxes.append(
                        (split, label_file.name, line_number, "wrong_format")
                    )
                    continue

                class_id, x, y, width, height = map(float, values)

                if not (0 <= x <= 1 and
                        0 <= y <= 1 and
                        0 < width <= 1 and
                        0 < height <= 1):

                    invalid_boxes.append(
                        (split, label_file.name, line_number, "invalid_coordinates")
                    )

print("Invalid annotations:", len(invalid_boxes))

if invalid_boxes:
    print("First few problems:")
    print(invalid_boxes[:10])

Invalid annotations: 0


In [21]:
import random
from PIL import Image

CLASS_NAMES = class_names

train_images_dir = EXTRACT_DIR / "train" / "images"
train_labels_dir = EXTRACT_DIR / "train" / "labels"

image_files = list(train_images_dir.glob("*.jpg"))

random.seed(SEED)
sample_images = random.sample(image_files, 6)

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for ax, image_path in zip(axes, sample_images):

    image = cv2.imread(str(image_path))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    height, width = image.shape[:2]

    label_path = train_labels_dir / f"{image_path.stem}.txt"

    if label_path.exists():
        with open(label_path, "r") as file:
            for line in file:

                values = line.strip().split()

                if len(values) != 5:
                    continue

                class_id, x_center, y_center, box_width, box_height = map(
                    float, values
                )

                x1 = int((x_center - box_width / 2) * width)
                y1 = int((y_center - box_height / 2) * height)
                x2 = int((x_center + box_width / 2) * width)
                y2 = int((y_center + box_height / 2) * height)

                class_id = int(class_id)

                cv2.rectangle(
                    image,
                    (x1, y1),
                    (x2, y2),
                    (255, 0, 0),
                    2
                )

                cv2.putText(
                    image,
                    CLASS_NAMES[class_id],
                    (x1, max(y1 - 8, 15)),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    (255, 0, 0),
                    2
                )

    ax.imshow(image)
    ax.set_title(image_path.name)
    ax.axis("off")

plt.tight_layout()
plt.show()

Output hidden; open in https://colab.research.google.com to view.

In [22]:
image_sizes = []

for image_path in image_files:
    image = cv2.imread(str(image_path))

    if image is not None:
        height, width = image.shape[:2]
        image_sizes.append((width, height))

sizes_df = pd.DataFrame(image_sizes, columns=["width", "height"])

print("Unique image dimensions:")
print(sizes_df.value_counts().head(10))

print("\nDimension statistics:")
print(sizes_df.describe())

Unique image dimensions:
width  height
1000   667       1374
       750        402
       668        155
667    1000        73
1000   664         72
       662         68
750    1000        63
1000   665         60
       563         47
       666         45
Name: count, dtype: int64

Dimension statistics:
             width       height
count  2816.000000  2816.000000
mean    978.997514   705.300781
std      77.448657    97.174243
min     562.000000   333.000000
25%    1000.000000   667.000000
50%    1000.000000   667.000000
75%    1000.000000   750.000000
max    1000.000000  1000.000000


In [23]:
corrupted_images = []

for split in ["train", "val", "test"]:

    image_dir = EXTRACT_DIR / split / "images"

    for image_path in image_dir.glob("*"):

        if image_path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue

        image = cv2.imread(str(image_path))

        if image is None:
            corrupted_images.append(str(image_path))

print("Unreadable images:", len(corrupted_images))

if corrupted_images:
    print(corrupted_images[:10])

Unreadable images: 0


In [24]:
from ultralytics import YOLO

MODEL_NAME = "yolo11n.pt"

model = YOLO(MODEL_NAME)

print(f"Loaded pretrained model: {MODEL_NAME}")

Loaded pretrained model: yolo11n.pt


In [25]:
TRAIN_CONFIG = {
    "data": str(yaml_path),
    "epochs": 40,
    "imgsz": 640,
    "batch": 16,
    "device": DEVICE,
    "workers": 2,
    "project": str(ARTIFACTS_DIR / "training"),
    "name": "vehicle_damage_yolo",
    "exist_ok": True,
    "patience": 8,
    "pretrained": True,
    "verbose": True,
}

TRAIN_CONFIG

{'data': '/content/vehicle_damage_detection/data/carddd/data.yaml',
 'epochs': 40,
 'imgsz': 640,
 'batch': 16,
 'device': 'cuda',
 'workers': 2,
 'project': '/content/vehicle_damage_detection/artifacts/training',
 'name': 'vehicle_damage_yolo',
 'exist_ok': True,
 'patience': 8,
 'pretrained': True,
 'verbose': True}

In [26]:
TRAIN_CONFIG = {
    "data": str(yaml_path),
    "epochs": 40,
    "imgsz": 640,
    "batch": 16,
    "device": DEVICE,
    "workers": 2,
    "project": str(ARTIFACTS_DIR / "training"),
    "name": "vehicle_damage_yolo",
    "exist_ok": True,
    "patience": 8,
    "pretrained": True,
    "verbose": True,
}

TRAIN_CONFIG

{'data': '/content/vehicle_damage_detection/data/carddd/data.yaml',
 'epochs': 40,
 'imgsz': 640,
 'batch': 16,
 'device': 'cuda',
 'workers': 2,
 'project': '/content/vehicle_damage_detection/artifacts/training',
 'name': 'vehicle_damage_yolo',
 'exist_ok': True,
 'patience': 8,
 'pretrained': True,
 'verbose': True}

In [27]:
import yaml

colab_data_config = {
    "path": str(EXTRACT_DIR),
    "train": "train/images",
    "val": "val/images",
    "test": "test/images",
    "nc": len(class_names),
    "names": class_names,
}

COLAB_YAML_PATH = CONFIG_DIR / "vehicle_damage_colab.yaml"

with open(COLAB_YAML_PATH, "w") as file:
    yaml.safe_dump(colab_data_config, file, sort_keys=False)

print("Created dataset configuration:")
print(COLAB_YAML_PATH)

print("\nConfiguration:")
print(colab_data_config)

Created dataset configuration:
/content/vehicle_damage_detection/artifacts/configs/vehicle_damage_colab.yaml

Configuration:
{'path': '/content/vehicle_damage_detection/data/carddd', 'train': 'train/images', 'val': 'val/images', 'test': 'test/images', 'nc': 6, 'names': ['dent', 'scratch', 'crack', 'glass shatter', 'lamp broken', 'tire flat']}


In [28]:
print(COLAB_YAML_PATH.read_text())

path: /content/vehicle_damage_detection/data/carddd
train: train/images
val: val/images
test: test/images
nc: 6
names:
- dent
- scratch
- crack
- glass shatter
- lamp broken
- tire flat



In [29]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
CUDA version: 12.8


In [30]:
TRAIN_CONFIG = {
    "data": str(COLAB_YAML_PATH),
    "epochs": 40,
    "imgsz": 640,
    "batch": 16,
    "device": DEVICE,
    "workers": 2,
    "project": str(ARTIFACTS_DIR / "training"),
    "name": "vehicle_damage_yolo",
    "exist_ok": True,
    "patience": 8,
    "pretrained": True,
    "verbose": True,
}

TRAIN_CONFIG

{'data': '/content/vehicle_damage_detection/artifacts/configs/vehicle_damage_colab.yaml',
 'epochs': 40,
 'imgsz': 640,
 'batch': 16,
 'device': 'cuda',
 'workers': 2,
 'project': '/content/vehicle_damage_detection/artifacts/training',
 'name': 'vehicle_damage_yolo',
 'exist_ok': True,
 'patience': 8,
 'pretrained': True,
 'verbose': True}

In [31]:
results = model.train(**TRAIN_CONFIG)

Ultralytics 8.4.126 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/vehicle_damage_detection/artifacts/configs/vehicle_damage_colab.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=40, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_

In [32]:
from pathlib import Path
import os

PROJECT_ROOT = Path("/content/vehicle_damage_detection")

TRAINING_DIR = PROJECT_ROOT / "artifacts" / "training" / "vehicle_damage_yolo"
BEST_MODEL = TRAINING_DIR / "weights" / "best.pt"
LAST_MODEL = TRAINING_DIR / "weights" / "last.pt"

print("Training directory:", TRAINING_DIR)
print("Training directory exists:", TRAINING_DIR.exists())

print("\nBest model:", BEST_MODEL)
print("Best model exists:", BEST_MODEL.exists())

print("Last model:", LAST_MODEL)
print("Last model exists:", LAST_MODEL.exists())

if BEST_MODEL.exists():
    print(f"\n✅ best.pt found: {BEST_MODEL.stat().st_size / (1024**2):.2f} MB")

if LAST_MODEL.exists():
    print(f"✅ last.pt found: {LAST_MODEL.stat().st_size / (1024**2):.2f} MB")

Training directory: /content/vehicle_damage_detection/artifacts/training/vehicle_damage_yolo
Training directory exists: True

Best model: /content/vehicle_damage_detection/artifacts/training/vehicle_damage_yolo/weights/best.pt
Best model exists: True
Last model: /content/vehicle_damage_detection/artifacts/training/vehicle_damage_yolo/weights/last.pt
Last model exists: True

✅ best.pt found: 5.20 MB
✅ last.pt found: 5.20 MB


In [33]:
from ultralytics import YOLO
from pathlib import Path

BEST_MODEL = Path(
    "/content/vehicle_damage_detection/artifacts/training/"
    "vehicle_damage_yolo/weights/best.pt"
)

model = YOLO(str(BEST_MODEL))

print("✅ Best model loaded")
print("Model:", BEST_MODEL)
print("Classes:", model.names)

✅ Best model loaded
Model: /content/vehicle_damage_detection/artifacts/training/vehicle_damage_yolo/weights/best.pt
Classes: {0: 'dent', 1: 'scratch', 2: 'crack', 3: 'glass shatter', 4: 'lamp broken', 5: 'tire flat'}


In [34]:
from pathlib import Path

DATA_YAML = "/content/vehicle_damage_detection/artifacts/configs/vehicle_damage_colab.yaml"

test_results = model.val(
    data=DATA_YAML,
    split="test",
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    plots=True,
    verbose=True
)

print("\n" + "="*60)
print("FINAL TEST SET EVALUATION")
print("="*60)

print(f"mAP50:       {test_results.box.map50:.4f}")
print(f"mAP50-95:    {test_results.box.map:.4f}")
print(f"Precision:   {test_results.box.mp:.4f}")
print(f"Recall:      {test_results.box.mr:.4f}")

Ultralytics 8.4.126 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,583,322 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 59.8±16.4 MB/s, size: 628.6 KB)
val: Scanning /content/vehicle_damage_detection/data/carddd/test/labels... 374 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 374/374 732.2it/s 0.5s
val: New cache created: /content/vehicle_damage_detection/data/carddd/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 24/24 2.8it/s 8.5s
                   all        374        785      0.714      0.677      0.713      0.565
                  dent        157        236      0.628      0.576      0.591      0.338
               scratch        183        307      0.624      0.554      0.594      0.328
                 crack         48         70      0.521      0.286      0.351      0.188
         glass shatter    

In [35]:
print("\n" + "="*70)
print("PER-CLASS TEST PERFORMANCE")
print("="*70)

class_names = model.names

for i, name in class_names.items():
    print(f"{i}: {name}")

print("\nPer-class mAP50:")
for i, value in enumerate(test_results.box.ap50):
    print(f"{class_names[i]:20s}: {value:.4f}")

print("\nPer-class mAP50-95:")
for i, value in enumerate(test_results.box.ap):
    print(f"{class_names[i]:20s}: {value:.4f}")


PER-CLASS TEST PERFORMANCE
0: dent
1: scratch
2: crack
3: glass shatter
4: lamp broken
5: tire flat

Per-class mAP50:
dent                : 0.5912
scratch             : 0.5942
crack               : 0.3508
glass shatter       : 0.9760
lamp broken         : 0.8437
tire flat           : 0.9249

Per-class mAP50-95:
dent                : 0.3379
scratch             : 0.3276
crack               : 0.1881
glass shatter       : 0.9149
lamp broken         : 0.7108
tire flat           : 0.9126


In [36]:
import json
from pathlib import Path

ARTIFACTS = Path("/content/vehicle_damage_detection/artifacts")
METRICS_DIR = ARTIFACTS / "metrics"
METRICS_DIR.mkdir(parents=True, exist_ok=True)

metrics = {
    "model": "YOLO11n",
    "task": "vehicle_damage_detection",
    "num_classes": 6,
    "classes": list(model.names.values()),
    "test_precision": float(test_results.box.mp),
    "test_recall": float(test_results.box.mr),
    "test_map50": float(test_results.box.map50),
    "test_map50_95": float(test_results.box.map),
    "image_size": 640,
    "best_model": str(BEST_MODEL)
}

metrics_path = METRICS_DIR / "test_metrics.json"

with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=4)

print("✅ Metrics saved to:")
print(metrics_path)

✅ Metrics saved to:
/content/vehicle_damage_detection/artifacts/metrics/test_metrics.json


In [37]:
TEST_IMAGES = "/content/vehicle_damage_detection/data/carddd/test/images"

INFERENCE_DIR = (
    "/content/vehicle_damage_detection/artifacts/inference"
)

predictions = model.predict(
    source=TEST_IMAGES,
    imgsz=640,
    conf=0.25,
    iou=0.7,
    device=0,
    save=True,
    save_txt=True,
    save_conf=True,
    project=INFERENCE_DIR,
    name="test_predictions",
    exist_ok=True,
    verbose=True
)

print("\n✅ Test inference completed")
print("Saved to:")
print(f"{INFERENCE_DIR}/test_predictions")


image 1/374 /content/vehicle_damage_detection/data/carddd/test/images/000012.jpg: 448x640 1 tire flat, 7.1ms
image 2/374 /content/vehicle_damage_detection/data/carddd/test/images/000015.jpg: 448x640 1 scratch, 10.2ms
image 3/374 /content/vehicle_damage_detection/data/carddd/test/images/000023.jpg: 448x640 1 dent, 1 scratch, 7.5ms
image 4/374 /content/vehicle_damage_detection/data/carddd/test/images/000033.jpg: 640x448 2 dents, 47.0ms
image 5/374 /content/vehicle_damage_detection/data/carddd/test/images/000040.jpg: 448x640 1 scratch, 8.1ms
image 6/374 /content/vehicle_damage_detection/data/carddd/test/images/000042.jpg: 448x640 5 dents, 1 glass shatter, 7.4ms
image 7/374 /content/vehicle_damage_detection/data/carddd/test/images/000044.jpg: 448x640 1 dent, 1 scratch, 2 cracks, 1 lamp broken, 7.3ms
image 8/374 /content/vehicle_damage_detection/data/carddd/test/images/000057.jpg: 448x640 1 tire flat, 8.3ms
image 9/374 /content/vehicle_damage_detection/data/carddd/test/images/000082.jpg: 4

In [38]:
import random
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

test_images = list(Path(TEST_IMAGES).glob("*"))

sample_images = random.sample(
    test_images,
    min(12, len(test_images))
)

results = model.predict(
    source=[str(x) for x in sample_images],
    imgsz=640,
    conf=0.25,
    device=0,
    verbose=False
)

for i, result in enumerate(results):
    annotated = result.plot()

    plt.figure(figsize=(10, 7))
    plt.imshow(annotated[:, :, ::-1])
    plt.axis("off")
    plt.title(f"Vehicle Damage Detection — Sample {i+1}")
    plt.show()

Output hidden; open in https://colab.research.google.com to view.

In [39]:
from pathlib import Path
import shutil

PRODUCTION_DIR = Path(
    "/content/vehicle_damage_detection/artifacts/model"
)

PRODUCTION_DIR.mkdir(parents=True, exist_ok=True)

production_model = PRODUCTION_DIR / "vehicle_damage_yolo11n_best.pt"

shutil.copy2(
    BEST_MODEL,
    production_model
)

print("✅ Production model saved:")
print(production_model)

print(
    f"Size: {production_model.stat().st_size / (1024**2):.2f} MB"
)

✅ Production model saved:
/content/vehicle_damage_detection/artifacts/model/vehicle_damage_yolo11n_best.pt
Size: 5.20 MB


In [40]:
from pathlib import Path
import shutil

PRODUCTION_DIR = Path(
    "/content/vehicle_damage_detection/artifacts/model"
)

PRODUCTION_DIR.mkdir(parents=True, exist_ok=True)

production_model = PRODUCTION_DIR / "vehicle_damage_yolo11n_best.pt"

shutil.copy2(
    BEST_MODEL,
    production_model
)

print("✅ Production model saved:")
print(production_model)

print(
    f"Size: {production_model.stat().st_size / (1024**2):.2f} MB"
)

✅ Production model saved:
/content/vehicle_damage_detection/artifacts/model/vehicle_damage_yolo11n_best.pt
Size: 5.20 MB


In [41]:
from ultralytics import YOLO

production_model_obj = YOLO(
    "/content/vehicle_damage_detection/artifacts/model/"
    "vehicle_damage_yolo11n_best.pt"
)

export_path = production_model_obj.export(
    format="onnx",
    imgsz=640,
    simplify=True,
    dynamic=False
)

print("\n✅ ONNX export completed")
print("Exported model:", export_path)

Ultralytics 8.4.126 🚀 Python-3.13.15 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
YOLO11n summary (fused): 101 layers, 2,583,322 parameters, 0 gradients, 6.4 GFLOPs

PyTorch: starting from '/content/vehicle_damage_detection/artifacts/model/vehicle_damage_yolo11n_best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 10, 8400) (5.2 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.13.15 environment at: /usr
Resolved 12 packages in 238ms
Prepared 4 packages in 1.87s
Installed 4 packages in 255ms
 + colorama==0.4.6
 + onnx==1.22.0
 + onnxruntime==1.29.0
 + onnxslim==0.1.96

requirements: AutoUpdate success ✅ 2.9s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with on

In [42]:
from pathlib import Path
import shutil
import os

PROJECT_ROOT = Path("/content/vehicle_damage_detection")
BACKUP_ROOT = PROJECT_ROOT / "artifacts" / "production_backup"

BACKUP_ROOT.mkdir(parents=True, exist_ok=True)

print("PROJECT ROOT:", PROJECT_ROOT)
print("BACKUP ROOT:", BACKUP_ROOT)

PROJECT ROOT: /content/vehicle_damage_detection
BACKUP ROOT: /content/vehicle_damage_detection/artifacts/production_backup


In [43]:
MODEL_SRC = PROJECT_ROOT / "artifacts" / "model"
MODEL_DST = BACKUP_ROOT / "model"

if MODEL_SRC.exists():
    shutil.copytree(
        MODEL_SRC,
        MODEL_DST,
        dirs_exist_ok=True
    )
    print("✅ Model artifacts copied")
else:
    print("❌ Model directory not found")

✅ Model artifacts copied


In [44]:
CONFIG_SRC = PROJECT_ROOT / "artifacts" / "configs"
CONFIG_DST = BACKUP_ROOT / "configs"

if CONFIG_SRC.exists():
    shutil.copytree(
        CONFIG_SRC,
        CONFIG_DST,
        dirs_exist_ok=True
    )
    print("✅ Configuration copied")
else:
    print("❌ Configuration directory not found")

✅ Configuration copied


In [45]:
INFERENCE_SRC = PROJECT_ROOT / "artifacts" / "inference"
INFERENCE_DST = BACKUP_ROOT / "inference"

if INFERENCE_SRC.exists():
    shutil.copytree(
        INFERENCE_SRC,
        INFERENCE_DST,
        dirs_exist_ok=True
    )
    print("✅ Inference artifacts copied")
else:
    print("⚠️ Inference directory not found — skipping")

✅ Inference artifacts copied


In [46]:
METRICS_SRC = PROJECT_ROOT / "artifacts" / "metrics"
METRICS_DST = BACKUP_ROOT / "metrics"

if METRICS_SRC.exists():
    shutil.copytree(
        METRICS_SRC,
        METRICS_DST,
        dirs_exist_ok=True
    )
    print("✅ Metrics copied")
else:
    print("⚠️ Metrics directory not found — skipping")

✅ Metrics copied


In [47]:
import json
from pathlib import Path

manifest = []

for path in BACKUP_ROOT.rglob("*"):
    if path.is_file():
        manifest.append({
            "path": str(path.relative_to(BACKUP_ROOT)),
            "size_mb": round(
                path.stat().st_size / (1024 * 1024),
                3
            )
        })

manifest_path = BACKUP_ROOT / "artifact_manifest.json"

with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=4)

print(f"✅ Manifest created: {manifest_path}")
print(f"Total files: {len(manifest)}")

✅ Manifest created: /content/vehicle_damage_detection/artifacts/production_backup/artifact_manifest.json
Total files: 750


In [48]:
import shutil
from pathlib import Path

backup_parent = BACKUP_ROOT.parent

zip_base = backup_parent / "vehicle_damage_production_artifacts"

zip_path = shutil.make_archive(
    str(zip_base),
    "zip",
    root_dir=BACKUP_ROOT
)

print("✅ ZIP created:")
print(zip_path)

size_gb = Path(zip_path).stat().st_size / (1024**3)
print(f"ZIP size: {size_gb:.3f} GB")

✅ ZIP created:
/content/vehicle_damage_detection/artifacts/vehicle_damage_production_artifacts.zip
ZIP size: 0.095 GB
